# Equitable Post-HCT Survival Prediction

Walkthrough of the ensemble described in *Post-Hematopoietic Cell
Transplantation Survival Predictions using Ensemble Machine Learning Models*
(ICIRCA 2026, doi:10.1109/ICIRCA69024.2026.11570591).

Every step calls into `src/hct_survival/`, so this notebook and the
`hct-survival train` command run identical code. Nothing here holds
credentials: see `scripts/download_data.py` for the data fetch.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import sys

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout)
pd.set_option("display.max_columns", 100)

from hct_survival.config import Config, CVConfig, Paths
from hct_survival.data import build_dataset

paths = Paths()
print("looking for data in", paths.data_raw)

## 1. Load and clean

Registry shorthand is expanded, null-like sentinels (`Not done`, `TBD`, `Missing disease status`) collapse to a single `Missing` level, and numeric columns are downcast without destroying `NaN`s.

In [ ]:
dataset = build_dataset(paths.train_csv, paths.test_csv)

print(f"{len(dataset.train):,} training rows, {len(dataset.features)} features "
      f"({len(dataset.categorical)} categorical, {len(dataset.numerical)} numerical)")
dataset.train[["efs", "efs_time", "race_group", "age_at_hct"]].head()

In [ ]:
dataset.train.groupby("race_group", observed=True).agg(
    n=("ID", "size"),
    event_rate=("efs", "mean"),
    median_followup=("efs_time", "median"),
).sort_values("n", ascending=False)

## 2. The target

The Kaplan-Meier survival probability at each observed time. The curve decreases, so a **larger** target value means a **shorter** duration — the target is a risk score, which is why the C-index negates it. Censored patients survived *at least* to their observed time, so a fixed offset is subtracted from theirs.

In [ ]:
from hct_survival.targets import kaplan_meier_target

time = dataset.train["efs_time"].to_numpy()
event = dataset.train["efs"].to_numpy()
y = kaplan_meier_target(time, event, censored_offset=0.10)

pd.DataFrame({"efs_time": time, "efs": event, "y": y}).groupby("efs").agg(
    n=("y", "size"), mean_y=("y", "mean"), min_y=("y", "min"), max_y=("y", "max")
)

## 3. Train the ensemble

Five base learners, ten folds stratified on event x race group, the Kaplan-Meier curve refitted inside each training fold so no validation outcome leaks into the label.

In [ ]:
from hct_survival.pipeline import run, save

config = Config(cv=CVConfig(n_splits=10, random_state=42))
result = run(config)
save(result)

## 4. Results

In [ ]:
result.leaderboard.style.format(precision=4).background_gradient(subset=["equity_score"], cmap="Greens")

In [ ]:
from hct_survival.ensemble import weights_frame

weights_frame(result.weights).style.format({"weight": "{:.3f}"})

## 5. Subgroup fairness

The competition metric is `mean(C_g) - std(C_g)` over race groups: a model is penalised for discriminating well on average while failing a subgroup.

In [ ]:
print(result.equity.to_string(index=False, float_format="%.4f"))
print()
for key, value in result.summary.items():
    print(f"{key:>22}: {value:.4f}" if isinstance(value, float) else f"{key:>22}: {value}")

## 6. Diagnostics

In [ ]:
%matplotlib inline
from hct_survival import plots

plots.model_comparison(result.leaderboard)
plots.equity_by_group(result.equity)
plots.kaplan_meier_by_group(result.dataset.train)
plots.residuals(result.target, result.oof_ensemble);

In [ ]:
vif = plots.variance_inflation(result.dataset.train, result.dataset.numerical)
vif.head(15)

## 7. Feature importance

Averaged across folds rather than read off the last fold's model, and computed on out-of-fold rows so the numbers are not inflated by the training data the model already saw.

In [ ]:
from hct_survival.encoders import encode_for_lightgbm
from hct_survival.models import build_estimator, make_folds

train_lgb, _ = encode_for_lightgbm(dataset.train, dataset.test, dataset.categorical)
folds = make_folds(dataset.train, config.cv)

gains = np.zeros(len(dataset.features))
for tr_idx, _ in folds:
    model = build_estimator("lgbm", config.model_params["lgbm"], 42, False)
    model.set_params(n_estimators=300)
    model.fit(train_lgb.iloc[tr_idx][dataset.features], y[tr_idx])
    gains += model.feature_importances_ / len(folds)

importance = (
    pd.DataFrame({"feature": dataset.features, "gain": gains})
    .sort_values("gain", ascending=False, ignore_index=True)
    .head(20)
)
importance

## 8. Submission

In [ ]:
submission = result.submission()
submission.to_csv("submission.csv", index=False)
submission.head()

---

### Reproducing the published configuration

The paper's numbers used the marginal target and an unweighted mean:

```python
legacy = Config(
    cv=CVConfig(n_splits=10, stratify=False),
    fold_safe_target=False,
    rank_average=False,
    optimise_weights=False,
)
legacy_result = run(legacy)
```